# Task 1: Product Label Prediction

This notebook trains **product label prediction** models for the five CBD/cannabis product forms:

- `flower`
- `oil`
- `gummies`
- `vape`
- `topical`

It assumes your repo already contains a **pair-level dataset** created from GPT extraction, where one original Reddit text can yield multiple `(product, sentiment)` pairs.

## What this notebook does

1. Loads the pair-level dataset
2. Filters to the 5 final product labels
3. Builds a **text-level multi-label dataset**
4. Splits by `text_id` to avoid leakage
5. Trains classical multi-label baselines:
   - TF-IDF + One-vs-Rest Logistic Regression
   - TF-IDF + One-vs-Rest LinearSVC
   - TF-IDF + Classifier Chains
6. Trains a transformer multi-label model (DistilBERT)
7. Evaluates with:
   - micro F1
   - macro F1
   - weighted F1
   - hamming loss
   - subset accuracy
   - per-label precision / recall / F1
   - per-label AUROC (when available)
8. Saves predictions, metrics, and figures

## Important note

This notebook is written to be **robust to small schema differences** in your repo. It tries to discover likely file paths and text columns automatically. You may still need to adjust one or two path names if your local filenames differ.


In [ ]:
# If needed, install once in your environment
# !pip install -U pandas numpy scikit-learn matplotlib transformers datasets torch accelerate evaluate joblib

In [ ]:
import os
import re
import json
import math
import random
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.multioutput import ClassifierChain
from sklearn.metrics import (
    f1_score,
    precision_recall_fscore_support,
    classification_report,
    hamming_loss,
    accuracy_score,
    roc_auc_score,
)
from sklearn.preprocessing import MultiLabelBinarizer

import joblib
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PRODUCT_LABELS = ['flower', 'oil', 'gummies', 'vape', 'topical']


## 1) Paths and file discovery

Update `PROJECT_ROOT` if needed. By default this notebook assumes it is placed somewhere inside or next to your cloned repo.


In [ ]:
# -------- CONFIG --------
# Change this if you want to force a specific repo location.
PROJECT_ROOT = Path('.').resolve()

# If notebook is outside the repo, uncomment and edit this:
# PROJECT_ROOT = Path('/path/to/Cannabis-Analysis').resolve()

OUTPUT_DIR = PROJECT_ROOT / 'product_label_prediction_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT =', PROJECT_ROOT)
print('OUTPUT_DIR    =', OUTPUT_DIR)


def find_first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return None

pair_candidates = [
    PROJECT_ROOT / 'data_processing' / 'data' / 'final' / 'pair_level_gpt41mini_clean.csv',
    PROJECT_ROOT / 'data' / 'final' / 'pair_level_gpt41mini_clean.csv',
    PROJECT_ROOT / 'pair_level_gpt41mini_clean.csv',
    PROJECT_ROOT / 'data_processing' / 'pair_level_gpt41mini_clean.csv',
]

PAIR_PATH = find_first_existing(pair_candidates)
print('PAIR_PATH =', PAIR_PATH)


In [ ]:
if PAIR_PATH is None:
    raise FileNotFoundError(
        'Could not find pair_level_gpt41mini_clean.csv. Update PROJECT_ROOT or add the correct path to pair_candidates.'
    )

pair_df = pd.read_csv(PAIR_PATH)
print(pair_df.shape)
pair_df.head()


## 2) Inspect schema and normalize columns

This code tries to detect the key columns automatically:
- text id column
- text column
- product column
- optional date column

If the text column is missing from the pair-level file, the notebook tries to merge from a likely cleaned source file.


In [ ]:
print(pair_df.columns.tolist())

In [ ]:
def find_col(cols, candidates):
    lower_map = {c.lower(): c for c in cols}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    # partial match fallback
    for c in cols:
        cl = c.lower()
        for cand in candidates:
            if cand.lower() in cl:
                return c
    return None

TEXT_ID_COL = find_col(pair_df.columns, ['text_id', 'id', 'reddit_id', 'source_id'])
PRODUCT_COL = find_col(pair_df.columns, ['product', 'product_label', 'label_product'])
SENTIMENT_COL = find_col(pair_df.columns, ['sentiment', 'label_sentiment'])
DATE_COL = find_col(pair_df.columns, ['date_utc', 'date', 'created_utc', 'created_at'])
TEXT_COL = find_col(pair_df.columns, ['text', 'full_text', 'body', 'comment', 'content'])

print('TEXT_ID_COL =', TEXT_ID_COL)
print('PRODUCT_COL =', PRODUCT_COL)
print('SENTIMENT_COL =', SENTIMENT_COL)
print('DATE_COL =', DATE_COL)
print('TEXT_COL =', TEXT_COL)


In [ ]:
# If text is missing from pair-level data, try merging from likely cleaned text files.
if TEXT_COL is None:
    text_candidates = [
        PROJECT_ROOT / 'data_processing' / 'data' / 'final' / 'unified_clean.csv',
        PROJECT_ROOT / 'data_processing' / 'data' / 'final' / 'unified_sentiment.csv',
        PROJECT_ROOT / 'data_processing' / 'data' / 'intermediate' / 'unified_clean.csv',
        PROJECT_ROOT / 'data' / 'final' / 'unified_clean.csv',
        PROJECT_ROOT / 'data' / 'final' / 'unified_sentiment.csv',
    ]
    text_path = find_first_existing(text_candidates)
    print('fallback text_path =', text_path)

    if text_path is None:
        raise ValueError(
            'No text column found in pair-level file and no fallback cleaned text file was found. '
            'Please provide a dataset with text or update text_candidates.'
        )

    text_df = pd.read_csv(text_path)
    print('fallback text_df shape =', text_df.shape)
    print(text_df.columns.tolist())

    text_id_fallback = find_col(text_df.columns, ['text_id', 'id', 'reddit_id', 'source_id'])
    text_col_fallback = find_col(text_df.columns, ['text', 'full_text', 'body', 'comment', 'content'])

    if text_id_fallback is None or text_col_fallback is None:
        raise ValueError('Fallback text file does not contain identifiable text_id/text columns.')

    pair_df = pair_df.merge(
        text_df[[text_id_fallback, text_col_fallback]].drop_duplicates(),
        left_on=TEXT_ID_COL,
        right_on=text_id_fallback,
        how='left'
    )
    TEXT_COL = text_col_fallback

print('Final TEXT_COL =', TEXT_COL)


## 3) Filter to the 5 product labels and clean text-level inputs

In [ ]:
# Basic cleanup
pair_df = pair_df.copy()
pair_df[PRODUCT_COL] = pair_df[PRODUCT_COL].astype(str).str.strip().str.lower()
if TEXT_COL is not None:
    pair_df[TEXT_COL] = pair_df[TEXT_COL].astype(str)
if TEXT_ID_COL is not None:
    pair_df[TEXT_ID_COL] = pair_df[TEXT_ID_COL].astype(str)

pair_df = pair_df[pair_df[PRODUCT_COL].isin(PRODUCT_LABELS)].copy()

if TEXT_COL is None or TEXT_ID_COL is None:
    raise ValueError('Need both text_id and text columns for text-level product prediction.')

pair_df = pair_df.dropna(subset=[TEXT_ID_COL, TEXT_COL, PRODUCT_COL])
pair_df = pair_df[pair_df[TEXT_COL].str.strip().ne('')].copy()

# Remove exact duplicate (text_id, product) rows
pair_df = pair_df.drop_duplicates(subset=[TEXT_ID_COL, PRODUCT_COL])

print(pair_df.shape)
pair_df.head()


In [ ]:
print(pair_df[PRODUCT_COL].value_counts())

## 4) Build text-level multi-label dataset

From pair-level rows, aggregate to one row per original text.


In [ ]:
# Aggregate products per text_id
agg_dict = {PRODUCT_COL: lambda s: sorted(set(s))}
# Keep the first text and first date (if present)
agg_dict[TEXT_COL] = 'first'
if DATE_COL is not None:
    agg_dict[DATE_COL] = 'first'

text_level = pair_df.groupby(TEXT_ID_COL, as_index=False).agg(agg_dict)
text_level = text_level.rename(columns={PRODUCT_COL: 'product_list', TEXT_COL: 'text'})

# Create binary label columns
mlb = MultiLabelBinarizer(classes=PRODUCT_LABELS)
Y = mlb.fit_transform(text_level['product_list'])
y_df = pd.DataFrame(Y, columns=PRODUCT_LABELS, index=text_level.index)
text_level = pd.concat([text_level, y_df], axis=1)
text_level['num_labels'] = text_level[PRODUCT_LABELS].sum(axis=1)
text_level['is_multilabel'] = (text_level['num_labels'] > 1).astype(int)

print(text_level.shape)
text_level.head()


In [ ]:
print('Texts:', len(text_level))
print('Multi-label texts:', int(text_level['is_multilabel'].sum()))
print('Percent multi-label:', round(100 * text_level['is_multilabel'].mean(), 2), '%')
print('
Per-label counts:')
print(text_level[PRODUCT_LABELS].sum().sort_values(ascending=False))

## 5) Train/validation/test split by `text_id`

This avoids leakage from using the same original text in more than one split.


In [ ]:
train_df, temp_df = train_test_split(
    text_level,
    test_size=0.30,
    random_state=SEED,
    shuffle=True,
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    shuffle=True,
)

print('train:', train_df.shape)
print('val:  ', val_df.shape)
print('test: ', test_df.shape)

for name, df_ in [('train', train_df), ('val', val_df), ('test', test_df)]:
    print(f'\n{name} label counts:')
    print(df_[PRODUCT_LABELS].sum())


In [ ]:
X_train = train_df['text'].astype(str).tolist()
X_val = val_df['text'].astype(str).tolist()
X_test = test_df['text'].astype(str).tolist()

y_train = train_df[PRODUCT_LABELS].values
y_val = val_df[PRODUCT_LABELS].values
y_test = test_df[PRODUCT_LABELS].values


## 6) Evaluation helpers

In [ ]:
def subset_accuracy(y_true, y_pred):
    return np.mean(np.all(y_true == y_pred, axis=1))


def evaluate_multilabel(y_true, y_pred, y_score=None, labels=PRODUCT_LABELS):
    metrics = {
        'micro_f1': f1_score(y_true, y_pred, average='micro', zero_division=0),
        'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'weighted_f1': f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'hamming_loss': hamming_loss(y_true, y_pred),
        'subset_accuracy': subset_accuracy(y_true, y_pred),
    }

    prfs = precision_recall_fscore_support(y_true, y_pred, average=None, zero_division=0)
    per_label = pd.DataFrame({
        'label': labels,
        'precision': prfs[0],
        'recall': prfs[1],
        'f1': prfs[2],
        'support': y_true.sum(axis=0),
    })

    auc_rows = []
    if y_score is not None:
        for i, label in enumerate(labels):
            try:
                auc = roc_auc_score(y_true[:, i], y_score[:, i])
            except Exception:
                auc = np.nan
            auc_rows.append({'label': label, 'auroc': auc})
        auc_df = pd.DataFrame(auc_rows)
        per_label = per_label.merge(auc_df, on='label', how='left')
    
    return metrics, per_label.sort_values('f1', ascending=False)


def print_results(model_name, metrics, per_label_df):
    print(f'===== {model_name} =====')
    for k, v in metrics.items():
        print(f'{k}: {v:.4f}')
    print('\nPer-label results:')
    display(per_label_df)


## 7) Baseline 1 — TF-IDF + One-vs-Rest Logistic Regression

In [ ]:
logreg_model = Pipeline([
    ('tfidf', TfidfVectorizer(
        lowercase=True,
        strip_accents='unicode',
        ngram_range=(1,2),
        min_df=3,
        max_df=0.95,
        sublinear_tf=True,
    )),
    ('clf', OneVsRestClassifier(
        LogisticRegression(
            max_iter=300,
            class_weight='balanced',
            solver='liblinear',
            random_state=SEED,
        )
    ))
])

logreg_model.fit(X_train, y_train)

logreg_val_score = np.column_stack([est.predict_proba(logreg_model.named_steps['tfidf'].transform(X_val))[:,1] for est in logreg_model.named_steps['clf'].estimators_])
logreg_test_score = np.column_stack([est.predict_proba(logreg_model.named_steps['tfidf'].transform(X_test))[:,1] for est in logreg_model.named_steps['clf'].estimators_])

logreg_val_pred = (logreg_val_score >= 0.5).astype(int)
logreg_test_pred = (logreg_test_score >= 0.5).astype(int)

logreg_metrics, logreg_per_label = evaluate_multilabel(y_test, logreg_test_pred, logreg_test_score)
print_results('TF-IDF + OVR Logistic Regression', logreg_metrics, logreg_per_label)


## 8) Baseline 2 — TF-IDF + One-vs-Rest LinearSVC

`LinearSVC` does not output calibrated probabilities by default, so AUROC is computed from decision scores.


In [ ]:
svc_model = Pipeline([
    ('tfidf', TfidfVectorizer(
        lowercase=True,
        strip_accents='unicode',
        ngram_range=(1,2),
        min_df=3,
        max_df=0.95,
        sublinear_tf=True,
    )),
    ('clf', OneVsRestClassifier(
        LinearSVC(class_weight='balanced', random_state=SEED)
    ))
])

svc_model.fit(X_train, y_train)

X_val_tfidf = svc_model.named_steps['tfidf'].transform(X_val)
X_test_tfidf = svc_model.named_steps['tfidf'].transform(X_test)

svc_val_score = np.column_stack([est.decision_function(X_val_tfidf) for est in svc_model.named_steps['clf'].estimators_])
svc_test_score = np.column_stack([est.decision_function(X_test_tfidf) for est in svc_model.named_steps['clf'].estimators_])

svc_val_pred = (svc_val_score >= 0).astype(int)
svc_test_pred = (svc_test_score >= 0).astype(int)

svc_metrics, svc_per_label = evaluate_multilabel(y_test, svc_test_pred, svc_test_score)
print_results('TF-IDF + OVR LinearSVC', svc_metrics, svc_per_label)


## 9) Baseline 3 — TF-IDF + Classifier Chains

This can help when labels co-occur.


In [ ]:
cc_model = Pipeline([
    ('tfidf', TfidfVectorizer(
        lowercase=True,
        strip_accents='unicode',
        ngram_range=(1,2),
        min_df=3,
        max_df=0.95,
        sublinear_tf=True,
    )),
    ('clf', ClassifierChain(
        LogisticRegression(
            max_iter=300,
            class_weight='balanced',
            solver='liblinear',
            random_state=SEED,
        ),
        order='random',
        random_state=SEED,
    ))
])

cc_model.fit(X_train, y_train)

# ClassifierChain supports predict_proba when the base estimator does.
cc_val_score = cc_model.predict_proba(X_val)
cc_test_score = cc_model.predict_proba(X_test)

cc_val_pred = (cc_val_score >= 0.5).astype(int)
cc_test_pred = (cc_test_score >= 0.5).astype(int)

cc_metrics, cc_per_label = evaluate_multilabel(y_test, cc_test_pred, cc_test_score)
print_results('TF-IDF + Classifier Chains (LogReg)', cc_metrics, cc_per_label)


## 10) Threshold tuning on the validation set

Instead of forcing `0.5` for every label, this searches a separate threshold per label to maximize label-wise F1 on the validation set.


In [ ]:
def tune_thresholds(y_true, y_score, labels=PRODUCT_LABELS, grid=None):
    if grid is None:
        grid = np.arange(0.10, 0.91, 0.05)
    thresholds = {}
    for i, label in enumerate(labels):
        best_thr, best_f1 = 0.5, -1
        for thr in grid:
            pred_i = (y_score[:, i] >= thr).astype(int)
            f1 = f1_score(y_true[:, i], pred_i, zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                best_thr = thr
        thresholds[label] = best_thr
    return thresholds


def apply_thresholds(y_score, thresholds, labels=PRODUCT_LABELS):
    preds = np.zeros_like(y_score, dtype=int)
    for i, label in enumerate(labels):
        preds[:, i] = (y_score[:, i] >= thresholds[label]).astype(int)
    return preds

# Tune on validation for the best classical model candidates
threshold_sets = {}
results_thresholded = {}

for name, val_score, test_score in [
    ('logreg', logreg_val_score, logreg_test_score),
    ('svc', svc_val_score, svc_test_score),
    ('cc', cc_val_score, cc_test_score),
]:
    thresholds = tune_thresholds(y_val, val_score)
    threshold_sets[name] = thresholds
    tuned_test_pred = apply_thresholds(test_score, thresholds)
    tuned_metrics, tuned_per_label = evaluate_multilabel(y_test, tuned_test_pred, test_score)
    results_thresholded[name] = (tuned_metrics, tuned_per_label)
    print('\n', name, thresholds)
    print_results(f'{name} (threshold tuned)', tuned_metrics, tuned_per_label)


## 11) Compare classical models

In [ ]:
rows = []
for name, metrics in [
    ('LogReg', logreg_metrics),
    ('LinearSVC', svc_metrics),
    ('ClassifierChain', cc_metrics),
    ('LogReg_tuned', results_thresholded['logreg'][0]),
    ('LinearSVC_tuned', results_thresholded['svc'][0]),
    ('ClassifierChain_tuned', results_thresholded['cc'][0]),
]:
    row = {'model': name}
    row.update(metrics)
    rows.append(row)

classical_summary = pd.DataFrame(rows).sort_values(['macro_f1', 'micro_f1'], ascending=False)
classical_summary


In [ ]:
best_classical_name = classical_summary.iloc[0]['model']
print('Best classical model:', best_classical_name)

## 12) Save classical artifacts

In [ ]:
classical_summary.to_csv(OUTPUT_DIR / 'classical_model_summary.csv', index=False)
logreg_per_label.to_csv(OUTPUT_DIR / 'logreg_per_label.csv', index=False)
svc_per_label.to_csv(OUTPUT_DIR / 'svc_per_label.csv', index=False)
cc_per_label.to_csv(OUTPUT_DIR / 'classifier_chain_per_label.csv', index=False)

joblib.dump(logreg_model, OUTPUT_DIR / 'logreg_multilabel.joblib')
joblib.dump(svc_model, OUTPUT_DIR / 'linearsvc_multilabel.joblib')
joblib.dump(cc_model, OUTPUT_DIR / 'classifier_chain_multilabel.joblib')

with open(OUTPUT_DIR / 'tuned_thresholds.json', 'w') as f:
    json.dump(threshold_sets, f, indent=2)

print('Saved classical outputs to', OUTPUT_DIR)


## 13) Plot classical macro/micro F1

In [ ]:
plot_df = classical_summary[['model', 'micro_f1', 'macro_f1']].copy()
ax = plot_df.set_index('model').plot(kind='bar', figsize=(10, 5))
ax.set_title('Classical model comparison for 5-label product prediction')
ax.set_ylabel('F1 score')
ax.set_xlabel('Model')
for c in ax.containers:
    ax.bar_label(c, fmt='%.3f', padding=2)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'classical_model_comparison_f1.png', dpi=200)
plt.show()


## 14) Transformer model — DistilBERT multi-label classifier

This section fine-tunes a transformer with 5 sigmoid outputs.

You can skip this section if you only want classical baselines.


In [ ]:
USE_TRANSFORMER = True  # Set to False to skip transformer training
HF_MODEL_NAME = 'distilbert-base-uncased'
MAX_LENGTH = 256
BATCH_SIZE = 16
NUM_EPOCHS = 3
LEARNING_RATE = 2e-5


In [ ]:
if USE_TRANSFORMER:
    import torch
    from datasets import Dataset
    from transformers import (
        AutoTokenizer,
        AutoModelForSequenceClassification,
        TrainingArguments,
        Trainer,
        DataCollatorWithPadding,
        set_seed,
    )

    set_seed(SEED)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print('device =', device)


In [ ]:
if USE_TRANSFORMER:
    train_hf = pd.DataFrame({'text': X_train})
    val_hf = pd.DataFrame({'text': X_val})
    test_hf = pd.DataFrame({'text': X_test})

    for i, label in enumerate(PRODUCT_LABELS):
        train_hf[label] = y_train[:, i]
        val_hf[label] = y_val[:, i]
        test_hf[label] = y_test[:, i]

    train_ds = Dataset.from_pandas(train_hf)
    val_ds = Dataset.from_pandas(val_hf)
    test_ds = Dataset.from_pandas(test_hf)

    tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_NAME)

    def tokenize_batch(batch):
        return tokenizer(batch['text'], truncation=True, max_length=MAX_LENGTH)

    train_ds = train_ds.map(tokenize_batch, batched=True)
    val_ds = val_ds.map(tokenize_batch, batched=True)
    test_ds = test_ds.map(tokenize_batch, batched=True)

    label2id = {label: i for i, label in enumerate(PRODUCT_LABELS)}
    id2label = {i: label for label, i in label2id.items()}

    def add_labels(example):
        example['labels'] = [float(example[label]) for label in PRODUCT_LABELS]
        return example

    train_ds = train_ds.map(add_labels)
    val_ds = val_ds.map(add_labels)
    test_ds = test_ds.map(add_labels)

    keep_cols = ['input_ids', 'attention_mask', 'labels']
    if 'token_type_ids' in train_ds.column_names:
        keep_cols.append('token_type_ids')

    train_ds.set_format(type='torch', columns=keep_cols)
    val_ds.set_format(type='torch', columns=keep_cols)
    test_ds.set_format(type='torch', columns=keep_cols)

    model = AutoModelForSequenceClassification.from_pretrained(
        HF_MODEL_NAME,
        num_labels=len(PRODUCT_LABELS),
        problem_type='multi_label_classification',
        id2label=id2label,
        label2id=label2id,
    )

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    def compute_metrics_transformer(eval_pred):
        logits, labels = eval_pred
        probs = 1 / (1 + np.exp(-logits))
        preds = (probs >= 0.5).astype(int)
        metrics, _ = evaluate_multilabel(labels.astype(int), preds, probs)
        return metrics

    training_args = TrainingArguments(
        output_dir=str(OUTPUT_DIR / 'distilbert_outputs'),
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=NUM_EPOCHS,
        weight_decay=0.01,
        evaluation_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='macro_f1',
        greater_is_better=True,
        logging_steps=50,
        seed=SEED,
        report_to='none',
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics_transformer,
    )

    trainer.train()


In [ ]:
if USE_TRANSFORMER:
    pred_out = trainer.predict(test_ds)
    logits = pred_out.predictions
    y_true_tf = np.array(test_hf[PRODUCT_LABELS])
    y_prob_tf = 1 / (1 + np.exp(-logits))
    y_pred_tf = (y_prob_tf >= 0.5).astype(int)

    tf_metrics, tf_per_label = evaluate_multilabel(y_true_tf, y_pred_tf, y_prob_tf)
    print_results('DistilBERT multi-label', tf_metrics, tf_per_label)


In [ ]:
if USE_TRANSFORMER:
    tf_thresholds = tune_thresholds(y_val, 1 / (1 + np.exp(-trainer.predict(val_ds).predictions)))
    y_pred_tf_tuned = apply_thresholds(y_prob_tf, tf_thresholds)
    tf_tuned_metrics, tf_tuned_per_label = evaluate_multilabel(y_true_tf, y_pred_tf_tuned, y_prob_tf)
    print('Transformer thresholds:', tf_thresholds)
    print_results('DistilBERT multi-label (threshold tuned)', tf_tuned_metrics, tf_tuned_per_label)


## 15) Save transformer outputs

In [ ]:
if USE_TRANSFORMER:
    tf_per_label.to_csv(OUTPUT_DIR / 'distilbert_per_label.csv', index=False)
    tf_tuned_per_label.to_csv(OUTPUT_DIR / 'distilbert_tuned_per_label.csv', index=False)
    pd.DataFrame([tf_metrics]).to_csv(OUTPUT_DIR / 'distilbert_metrics.csv', index=False)
    pd.DataFrame([tf_tuned_metrics]).to_csv(OUTPUT_DIR / 'distilbert_tuned_metrics.csv', index=False)

    test_pred_df = test_df[[TEXT_ID_COL, 'text']].copy().reset_index(drop=True)
    for i, label in enumerate(PRODUCT_LABELS):
        test_pred_df[f'true_{label}'] = y_true_tf[:, i]
        test_pred_df[f'prob_{label}'] = y_prob_tf[:, i]
        test_pred_df[f'pred_{label}'] = y_pred_tf_tuned[:, i]
    test_pred_df.to_csv(OUTPUT_DIR / 'distilbert_test_predictions.csv', index=False)

    trainer.save_model(str(OUTPUT_DIR / 'distilbert_best_model'))
    tokenizer.save_pretrained(str(OUTPUT_DIR / 'distilbert_best_model'))

    with open(OUTPUT_DIR / 'distilbert_thresholds.json', 'w') as f:
        json.dump(tf_thresholds, f, indent=2)

    print('Saved transformer artifacts.')


## 16) Final comparison table

In [ ]:
final_rows = classical_summary.to_dict(orient='records')
if USE_TRANSFORMER:
    final_rows.append({'model': 'DistilBERT', **tf_metrics})
    final_rows.append({'model': 'DistilBERT_tuned', **tf_tuned_metrics})

final_summary = pd.DataFrame(final_rows).sort_values(['macro_f1', 'micro_f1'], ascending=False)
final_summary.to_csv(OUTPUT_DIR / 'final_model_summary.csv', index=False)
final_summary


In [ ]:
ax = final_summary.set_index('model')[['micro_f1', 'macro_f1']].plot(kind='bar', figsize=(11, 5))
ax.set_title('Final comparison: 5-label product prediction')
ax.set_ylabel('F1 score')
ax.set_xlabel('Model')
for c in ax.containers:
    ax.bar_label(c, fmt='%.3f', padding=2)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'final_model_comparison_f1.png', dpi=200)
plt.show()


## 17) Error analysis helpers

These cells help inspect false negatives and false positives for each label.


In [ ]:
# Choose which predictions to analyze
if USE_TRANSFORMER:
    analysis_probs = y_prob_tf
    analysis_pred = y_pred_tf_tuned
    analysis_true = y_true_tf
    analysis_text_df = test_df[[TEXT_ID_COL, 'text']].reset_index(drop=True).copy()
    analysis_model_name = 'DistilBERT_tuned'
else:
    analysis_probs = svc_test_score
    analysis_pred = apply_thresholds(svc_test_score, threshold_sets['svc'])
    analysis_true = y_test
    analysis_text_df = test_df[[TEXT_ID_COL, 'text']].reset_index(drop=True).copy()
    analysis_model_name = 'LinearSVC_tuned'

print('Analyzing:', analysis_model_name)


In [ ]:
def collect_errors(text_df, y_true, y_pred, y_score, labels=PRODUCT_LABELS, top_n=20):
    out = {}
    for i, label in enumerate(labels):
        df = text_df.copy()
        df['true'] = y_true[:, i]
        df['pred'] = y_pred[:, i]
        df['score'] = y_score[:, i]

        fn = df[(df['true'] == 1) & (df['pred'] == 0)].sort_values('score', ascending=True).head(top_n)
        fp = df[(df['true'] == 0) & (df['pred'] == 1)].sort_values('score', ascending=False).head(top_n)
        out[label] = {'false_negatives': fn, 'false_positives': fp}
    return out

error_examples = collect_errors(analysis_text_df, analysis_true, analysis_pred, analysis_probs)


In [ ]:
label_to_inspect = 'vape'  # change as needed
print('False negatives for', label_to_inspect)
display(error_examples[label_to_inspect]['false_negatives'][[TEXT_ID_COL, 'score', 'text']])

print('False positives for', label_to_inspect)
display(error_examples[label_to_inspect]['false_positives'][[TEXT_ID_COL, 'score', 'text']])


## 18) Save the text-level dataset used for product prediction

This is useful for the report and for reusing the same split later.


In [ ]:
text_level.to_csv(OUTPUT_DIR / 'text_level_5label_dataset.csv', index=False)
train_df.to_csv(OUTPUT_DIR / 'train_text_level.csv', index=False)
val_df.to_csv(OUTPUT_DIR / 'val_text_level.csv', index=False)
test_df.to_csv(OUTPUT_DIR / 'test_text_level.csv', index=False)

print('Saved text-level datasets and splits.')


## 19) What to do next

After finishing **product label prediction**, your next notebook should do **pair-level multiclass sentiment prediction**:

- Input: `text + target product`
- Output: `positive / negative / neutral`

Recommended input format for the next task:

```text
[PRODUCT=vape] this oil helps me sleep but vape makes me cough
```

That lets the sentiment model focus on the correct product within the same text.
